# Forecasting trained-pilot catalogue selections

Predict A01's selected skill at exactly **+1, +2 and +5 seconds** using only B01's observation history. These are policy catalogue labels; several names share executed controls and need not denote distinct physical manoeuvres.

**Run All reads saved development results and demonstrates inference.** Preparation and training switches below default to off. The 45-group test partition remains reserved. The previous motion experiment is preserved.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "bvr_behavior_prediction").is_dir())
sys.path.insert(0, str(ROOT))
DATA = ROOT / "artifacts/skill_forecasting/data_v1"
EXPERIMENT = ROOT / "artifacts/skill_forecasting/initial_v1"
CONFIG = ROOT / "configs/training/skill_forecasting.json"
SCRIPT = ROOT / "scripts/train_skill_forecaster.py"
PREPARE_DATA = False
TRAIN_MODELS = False
ARCHITECTURES = ["gru", "transformer"]

def run(command, *arguments):
    subprocess.run([sys.executable, "-X", "utf8", str(SCRIPT), command,
                    "--data", str(DATA), "--output", str(EXPERIMENT),
                    "--config", str(CONFIG), *arguments], cwd=ROOT, check=True)

if PREPARE_DATA:
    run("prepare")
if TRAIN_MODELS:
    run("baseline")
    for architecture in ARCHITECTURES:
        run("train", "--architecture", architecture)
        run("verify", "--architecture", architecture)
    run("report")

assert (DATA / "data_manifest.json").exists(), "Prepare data using the documented CLI first."
manifest = json.loads((DATA / "data_manifest.json").read_text())
display(pd.DataFrame(manifest["counts"]).T)

## Label alignment and support

A 6-second origin uses observations through 6 seconds and decisions at 7, 8 and 11 seconds. Missing decisions are masked independently; the final choice is never extended into an unrecorded future. Fifty-one model frames span five seconds; the backward acceleration features require 56 raw frames. Splits match the motion experiment exactly, and preprocessing is fitted on training frames only.

In [ ]:
audit = json.loads((DATA / "alignment_audit.json").read_text())
display(audit)
support = pd.read_csv(DATA / "label_support.csv")
population_support = support.query("pilot_id == 'population'")
display(population_support.pivot(index="skill", columns=["split", "horizon_s"], values="count"))
raw_support = pd.read_csv(DATA / "decision_support.csv")
display(raw_support.groupby(["skill", "split"])["count"].sum().unstack(fill_value=0))

## Validation comparison

Metrics weight valid forecast windows equally. Catalogue macro-F1 averages all 33 labels (undefined F1 is zero); supported macro-F1 averages only labels with truth support in that subset. Rare-class estimates are unstable, and overlapping windows are correlated. The frequency baseline is fitted on training labels separately at each horizon.

In [ ]:
paths = sorted(EXPERIMENT.glob("*/validation_metrics.csv"))
assert paths, "Run the frequency baseline and training commands first."
metrics = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
population = metrics.query("pilot_id == 'population' and cohort == 'all'")
columns = ["model", "horizon_s", "count", "supported_classes", "top1_accuracy",
           "top3_accuracy", "macro_f1_catalogue", "macro_f1_supported", "nll"]
display(population[columns].round(4))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, measure in zip(axes, ["top1_accuracy", "top3_accuracy", "macro_f1_catalogue"]):
    population.pivot(index="horizon_s", columns="model", values=measure).plot(
        kind="bar", ax=ax, title=measure, rot=0)
    ax.set_xlabel("Forecast horizon (seconds)")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Training and checkpoint verification

Checkpoint selection uses mean unweighted validation negative log-likelihood across horizons. Training defaults to unweighted cross-entropy. Optional capped square-root class weights are a separate experiment.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for architecture in ARCHITECTURES:
    directory = EXPERIMENT / architecture
    if not (directory / "history.csv").exists():
        continue
    history = pd.read_csv(directory / "history.csv")
    ax.plot(history.epoch, history.validation_nll, label=architecture)
    print(architecture, json.loads((directory / "summary.json").read_text()))
    verification = directory / "verification.json"
    if verification.exists():
        print("Checkpoint verification:", json.loads(verification.read_text()))
ax.set(xlabel="Epoch", ylabel="Validation NLL")
ax.legend()
plt.show()

## Per-pilot and changed/unchanged results

Pilot IDs identify evaluation subsets only. The change analysis uses the privileged true selection at the origin, never an encoder input. It compares endpoint labels; intermediate switches and returns count as the same endpoint label. This is not an operational persistence baseline.

In [ ]:
display(metrics.query("pilot_id != 'population' and cohort == 'all'").pivot(
    index=["pilot_id", "horizon_s"], columns="model", values="top1_accuracy").round(4))
display(metrics.query("pilot_id == 'population' and cohort in ['same', 'changed']")[
    ["model", "horizon_s", "cohort", "count", "top1_accuracy", "top3_accuracy",
     "macro_f1_catalogue"]].round(4))

## Confusion matrices and per-class scores

Rows are true labels and columns are predicted labels. Row normalization makes rare labels visible but can be misleading when based on only one or two cases; use the accompanying support counts.

In [ ]:
for architecture in ARCHITECTURES:
    directory = EXPERIMENT / architecture
    if not (directory / "confusion_matrices.npz").exists():
        continue
    matrices = np.load(directory / "confusion_matrices.npz")
    names = matrices["classes"]
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    for ax, horizon in zip(axes, [1, 2, 5]):
        counts = matrices[f"population_{horizon}s"]
        denominator = counts.sum(axis=1, keepdims=True)
        normalized = np.divide(counts, denominator, out=np.zeros_like(counts, dtype=float),
                               where=denominator > 0)
        ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(range(len(names)), names, rotation=90, fontsize=6)
        ax.set_yticks(range(len(names)), names, fontsize=6)
        ax.set(title=f"{architecture}, +{horizon}s", xlabel="Predicted", ylabel="True")
    plt.tight_layout()
    plt.show()
    scores = pd.read_csv(directory / "per_class_metrics.csv")
    display(scores.query("pilot_id == 'population'").sort_values(
        ["horizon_s", "support"], ascending=[True, False]).round(4))

## Observation-only checkpoint inference

Load one validation history in the original 51-feature observation scale. The inference interface accepts only consecutive past/current observations; target decisions, pilot identity and future observations are not supplied.

In [ ]:
from bvr_behavior_prediction.skill_forecasting.inference import SkillPredictor

available = [a for a in ARCHITECTURES if (EXPERIMENT / a / "best.pt").exists()]
if available:
    architecture = available[0]
    samples = pd.read_parquet(DATA / "samples.parquet")
    origin = samples.loc[samples.split.eq("validation")].iloc[0]
    observations = pd.read_parquet(
        Path(manifest["corpus"]) / "observations" / f"{origin.pilot_id}.parquet",
        filters=[("episode_id", "==", origin.episode_id), ("step", "<=", int(origin.step))])
    raw = observations.sort_values("step")[[f"obs_{i:02}" for i in range(51)]].to_numpy()
    predictor = SkillPredictor(EXPERIMENT / architecture / "best.pt")
    forecast = predictor.predict(raw)
    display(pd.DataFrame(forecast["top3_skills"], index=forecast["horizons_s"],
                         columns=["First", "Second", "Third"]).rename_axis("Horizon (s)"))
    print("Probability shape:", forecast["probabilities"].shape)
    print("Report:", EXPERIMENT / "report.html")

The reserved test set has not been evaluated. These results concern new starting scenarios for the known ten-pilot population. Unseen-pilot generalisation, realistic sensor noise and probability calibration remain untested. See [the skill-forecasting guide](../docs/skill_forecasting.md) for commands, provenance and interpretation.